# CenterSpeed 학습 (WandB로 train/val loss 시각화)
- 이 노트북은 train loss, val loss를 wandb로 실시간 시각화합니다.
- 나머지 기능(데이터셋, 모델, loss 등)은 기존과 동일하게 유지됩니다.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys, os, random, datetime
from torch.utils.data import DataLoader, random_split
import wandb

current_dir = os.path.dirname(os.path.abspath(''))
two_up_dir = os.path.dirname(os.path.dirname(current_dir))
sys.path.append(two_up_dir)

# from TinyCenterSpeed.src.models.resnet import *
from TinyCenterSpeed.src.models.CenterSpeed import *
from TinyCenterSpeed.dataset.CenterSpeed_dataset import *
from TinyCenterSpeed.src.models.losses import *
from train import *



%env "WANDB_NOTEBOOK_NAME" "train_CenterSpeed_wandb.ipynb"
wandb.login()

env: "WANDB_NOTEBOOK_NAME"="train_CenterSpeed_wandb.ipynb"


wandb: Currently logged in as: whdaudpark (whdaudpark-dongguk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# transform = transforms.Compose([RandomRotation(45),
#                                 RandomFlip(0.5)])


# set = CenterSpeedDataset('/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT', transform=transform, dense=True)
# set.seq_len = 2

# # x축과 y축 방향의 가우시안 표준편차(σ) 설정, 값이 클수록 넓게 퍼짐 -> GT heatmap만들 때 쓰임 
# # 객체 주변 0.9 픽셀로 점 찍음 
# # 2 ~ 5 정도로 하자 
# set.sx = 2
# set.sy = 2
# set.change_image_size(128) 
# print("dataset len:", len(set))
# # set.change_pixel_size(0.1)


In [ ]:
# transform = transforms.Compose([RandomRotation(45), RandomFlip(0.5)])
# set = CenterSpeedDataset('/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT',
#                         transform=transform, dense=True, preload=False)

# def worker_init_fn(_):
#     # 워커 과도한 스레드 사용 방지(과구독 방지)
#     try:
#         import torch, numpy as np, os
#         torch.set_num_threads(1)
#         os.environ.setdefault("OMP_NUM_THREADS", "1")
#         os.environ.setdefault("MKL_NUM_THREADS", "1")
#     except Exception:
#         pass

# num_workers = 15  # 16코어 기준
# loader = DataLoader(
#     dataset=set,                    # 네 데이터셋
#     batch_size=32,                  # VRAM에 맞게 32~64 시도
#     shuffle=True,
#     num_workers=num_workers,
#     pin_memory=True,
#     persistent_workers=True,
#     prefetch_factor=4,
#     worker_init_fn=worker_init_fn,
# )

# # x축과 y축 방향의 가우시안 표준편차(σ) 설정, 값이 클수록 넓게 퍼짐 -> GT heatmap만들 때 쓰임 
# # 객체 주변 0.9 픽셀로 점 찍음 
# # 2 ~ 5 정도로 하자 

# set.sx = 2
# set.sy = 2
# set.change_image_size(128) 
# print("dataset len:", len(set))
# # set.change_pixel_size(0.1)
# print(os.cpu_count())

[Lazy] Indexed 3 files, total usable samples: 34330
Image size changed to: 128 origin_offset: 6.4
dataset len: 34330
16


In [ ]:
# # 데이터셋 분할 및 DataLoader 생성
# train_size = int(len(set) * 0.75)
# print(f"train_size: {train_size}")
# val_size = int(len(set) * 0.2)
# test_size = len(set) - (train_size + val_size)

# train_dataset, val_dataset, test_dataset = random_split(set, [train_size, val_size, test_size])

# batch_size = 32

# training_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# validation_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
# testing_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)


In [ ]:
# # 재현 가능한 split
# g = torch.Generator().manual_seed(42)

# N = len(set)  # 'set' 대신 'dataset' 같은 이름 권장
# train_size = int(N * 0.75)
# val_size   = int(N * 0.20)
# test_size  = N - (train_size + val_size)
# print(f"train/val/test = {train_size}/{val_size}/{test_size}")

# train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
#     set, [train_size, val_size, test_size], generator=g
# )

# batch_size = 32

# # 워커 튜닝 (CPU 16개 기준)
# num_workers = 15
# def worker_init_fn(_):
#     try:
#         torch.set_num_threads(1)
#         os.environ.setdefault("OMP_NUM_THREADS", "1")
#         os.environ.setdefault("MKL_NUM_THREADS", "1")
#     except Exception:
#         pass

# training_loader = DataLoader(
#     train_dataset, batch_size=batch_size, shuffle=True,
#     num_workers=num_workers, pin_memory=True,
#     persistent_workers=True, prefetch_factor=4,
#     drop_last=True, worker_init_fn=worker_init_fn
# )

# validation_loader = DataLoader(
#     val_dataset, batch_size=batch_size, shuffle=False,
#     num_workers=num_workers, pin_memory=True,
#     persistent_workers=True, prefetch_factor=4,
#     drop_last=False, worker_init_fn=worker_init_fn
# )

# testing_loader = DataLoader(
#     test_dataset, batch_size=1, shuffle=False,
#     num_workers=num_workers, pin_memory=True,
#     persistent_workers=True, prefetch_factor=4,
#     worker_init_fn=worker_init_fn
# )



train/val/test = 25747/6866/1717


In [2]:
# ----- Transform -----
transform = transforms.Compose([
    RandomRotation(45),
    RandomFlip(0.5)   # mode='horizontal' 기본
])

# ----- Dataset (lazy load 권장: preload=False) -----
dataset = CenterSpeedDataset(
    dataset_path="/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT",
    transform=None,   # 학습에는 증강 적용
    dense=True,
    preload=False          # 대용량에서 메모리 절약
)
# 가우시안 폭 설정 및 이미지 크기
dataset.sx = 2
dataset.sy = 2
dataset.change_image_size(128)

print("dataset len:", len(dataset))
print("CPU count:", os.cpu_count())

# ----- 재현 가능한 split -----
g = torch.Generator().manual_seed(42)
N = len(dataset)
train_size = int(N * 0.75)
val_size   = int(N * 0.20)
test_size  = N - (train_size + val_size)
print(f"train/val/test = {train_size}/{val_size}/{test_size}")

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=g)

# ----- DataLoader 튜닝 (CPU 16코어 기준) -----
def worker_init_fn(_):
    try:
        torch.set_num_threads(1)
        os.environ.setdefault("OMP_NUM_THREADS", "1")
        os.environ.setdefault("MKL_NUM_THREADS", "1")
    except Exception:
        pass

batch_size = 32
num_workers = 15  # 16코어라면 14~15 권장

training_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=True,
    persistent_workers=True, prefetch_factor=4,
    drop_last=True, worker_init_fn=worker_init_fn
)

validation_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=True,
    persistent_workers=True, prefetch_factor=4,
    drop_last=False, worker_init_fn=worker_init_fn
)

testing_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=num_workers, pin_memory=True,
    persistent_workers=True, prefetch_factor=4,
    worker_init_fn=worker_init_fn
)



[Lazy] Indexed 6 files, total usable samples: 46405
Image size changed to: 128 origin_offset: 6.4
dataset len: 46405
CPU count: 16
train/val/test = 34803/9281/2321


In [3]:
# ==== Cell 2: Device, Model, Optimizer ====

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

net = CenterSpeedDense(image_size=128).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=5e-4, weight_decay=1e-4)

# (선택) 모델 입력 채널 확인
try:
    print("net.input_channels:", net.input_channels)
except Exception:
    pass


device: cuda
net.input_channels: 4


In [4]:
import torch.nn.functional as F

def dense_loss(output, gt_heatmap, gt_dense, is_free,
               alpha=0.9, tau=0.2, pos_weight=4.0):
    """
    output: [B,4,H,W] (0: heatmap logit, 1: vx, 2: vy, 3: yaw)
    gt_heatmap: [B,H,W] in [0,1]
    gt_dense:   [B,3,H,W]  # vx,vy,yaw (가급적 표준화)
    """
    logit = output[:,0]       # [B,H,W]
    preds = output[:,1:]      # [B,3,H,W]

    # Heatmap: BCEWithLogits(mean)
    hm_loss = F.binary_cross_entropy_with_logits(
        logit, gt_heatmap, pos_weight=torch.tensor([pos_weight], device=output.device),
        reduction='mean'
    )

    # Dense: heatmap>tau만 평균 MSE
    with torch.no_grad():
        mask = (gt_heatmap > tau).float().unsqueeze(1)  # [B,1,H,W]
    mse = ((preds - gt_dense)**2 * mask)
    denom = mask.sum(dim=(2,3), keepdim=True).clamp_min(1.0)
    dense = (mse.sum(dim=(2,3), keepdim=True) / denom).mean()

    return alpha * hm_loss + (1 - alpha) * dense


In [ ]:
# # 모델, optimizer, loss 함수 정의
# device = torch.device("cuda" if torch.cuda.is_available()else "cpu")
# print(device)

# net = CenterSpeedDense(image_size=128).to(device)
# optimizer = torch.optim.Adam(net.parameters(), lr=5e-4, weight_decay=1e-4)

# print(net.input_channels)

# # is_free : LiDAR 기반으로 장애물이 없는 곳 -> 1, 장애물 있는 곳 -> 0
# def dense_loss(output, gt_heatmap, gt_dense_data, is_free, alpha=0.9, decay=1):
#     device = output.device
#     gt_heatmap = gt_heatmap.to(device, dtype=torch.float32)
#     gt_dense_data = gt_dense_data.to(device, dtype=torch.float32)
#     loss = 0
#     batch_size = output.shape[0]
    
#     w = gt_heatmap
#     loss += (alpha * (1+w)* (output[:,:,:,0].unsqueeze(-1) - gt_heatmap)**2).sum()
#     loss += ((1-alpha) * (1+w)* (output[:,:,:,1:] - gt_dense_data)**2).sum()
#     return loss/ batch_size



cuda
4


In [5]:
from tqdm.auto import tqdm
import time

def train_and_validate(EPOCHS):
    losses = []
    val_losses = []
    epoch_durations = []
    start_all = time.time()

    for epoch in range(EPOCHS):
        print(f"Epoch: {epoch}")
        # net.to(device)
        net.train()
        running_loss = 0.0
        start_epoch = time.time()

        # ── Train 진행바 ──────────────────────────────────────────
        pbar = tqdm(training_loader, desc="Train", leave=False, dynamic_ncols=True)
        last = time.time()
        print(start_epoch-last)
        for i, batch in enumerate(pbar, 1):
            inputs, gts, data_, dense_data, is_free = batch
            inputs     = inputs.to(device)
            gts        = gts.to(device)
            data_      = data_.to(device)
            # dense_data shape 맞추기 [B, H, W, 3] -> [B, 3, H, W]
            if dense_data.dim() == 4 and dense_data.shape[1] != 3:
                dense_data = dense_data.permute(0, 3, 1, 2)
            dense_data = dense_data.to(device)
            is_free    = is_free.to(device)

            optimizer.zero_grad(set_to_none=True)
            output = net(inputs)
            loss = dense_loss(output, gts, dense_data, is_free)
            
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # tqdm에 현재 배치 loss/ETA 표시
            now = time.time()
            batch_time = now - last
            last = now
            rem = len(training_loader) - i
            eta_batch = rem * batch_time
            pbar.set_postfix(loss=f"{loss.item():.4f}", eta=f"{eta_batch/60:.1f}m")

        avg_loss = running_loss / max(len(training_loader), 1)
        losses.append(avg_loss)

        # ── Validation 진행바(선택) ───────────────────────────────
        net.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            vpbar = tqdm(validation_loader, desc="Val", leave=False, dynamic_ncols=True)
            for batch in vpbar:
                inputs, gts, data_, dense_data, is_free = batch
                inputs     = inputs.to(device)
                gts        = gts.to(device)
                data_      = data_.to(device)
                # dense_data shape 맞추기 [B, H, W, 3] -> [B, 3, H, W]
                if dense_data.dim() == 4 and dense_data.shape[1] != 3:
                    dense_data = dense_data.permute(0, 3, 1, 2)
                dense_data = dense_data.to(device)
                is_free    = is_free.to(device)

                output = net(inputs)
                val_loss = dense_loss(output, gts, dense_data, is_free)

                running_val_loss += val_loss.item()
                vpbar.set_postfix(loss=f"{val_loss.item():.4f}")

        avg_val_loss = running_val_loss / max(len(validation_loader), 1)
        val_losses.append(avg_val_loss)

        # ── 10에폭마다 모델 저장 ───────────────────────────────
        SAVE_EVERY = 10
        if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
            save_path = f"/home/harry/ros2_ws/src/TinyCenterSpeed/src/trained_models/redbull_0812_no_intensity_trans_adam_wd_epoch_{epoch+1}.pt"
            torch.save(net.state_dict(), save_path)
            print(f"✅ 모델 저장 완료: {save_path}")

        # ── 에폭 ETA/로그 ────────────────────────────────────────
        epoch_sec = time.time() - start_epoch
        epoch_durations.append(epoch_sec)
        avg_epoch_sec = sum(epoch_durations) / len(epoch_durations)
        remaining_epochs = EPOCHS - (epoch + 1)
        eta_min = max(0.0, remaining_epochs * avg_epoch_sec) / 60.0

        if wandb.run is not None:
            wandb.log({
                "epoch": epoch,
                "train/loss": avg_loss,
                "val/loss": avg_val_loss,
                "time/epoch_sec": epoch_sec,
                "time/eta_minutes": eta_min
            }, commit=True)

        print(f'  train_loss={avg_loss:.6f} | val_loss={avg_val_loss:.6f} | epoch_sec={epoch_sec:.2f}s | ETA={eta_min:.1f} min')

    return losses, val_losses


In [ ]:


# from tqdm.auto import tqdm
# import time


# def train_and_validate(EPOCHS):
#     losses = []
#     val_losses = []
#     epoch_durations = []
#     start_all = time.time()

#     for epoch in range(EPOCHS):
#         print(f"Epoch: {epoch}")
#         # net.to(device)
#         net.train()
#         running_loss = 0.0
#         start_epoch = time.time()

#         # ── Train 진행바 ──────────────────────────────────────────
#         pbar = tqdm(training_loader, desc="Train", leave=False, dynamic_ncols=True)
#         last = time.time()
#         load_start = time.time()
#         for i, batch in enumerate(pbar, 1):
#             load_end = time.time()
#             if i > 1:
#                 print(f"[Train] Batch {i-1} 데이터로딩 시간: {load_end - load_start:.4f}초")
#             load_start = time.time()
            


In [6]:
# W&B 설정 및 학습 실행
wandb.init(project='CenterSpeed', name='train_CenterSpeed_wandb_0812_17_15', config={'batch_size': 32, 'lr': 5e-4, 'epochs': 100})
losses, val_losses = train_and_validate(EPOCHS=100)
wandb.finish()

Epoch: 0


Train:   0%|          | 0/1087 [00:00<?, ?it/s]

-0.24267148971557617


Val:   0%|          | 0/291 [00:00<?, ?it/s]

  train_loss=0.608563 | val_loss=10056.595705 | epoch_sec=26.79s | ETA=44.2 min
Epoch: 1


...

Train:   0%|          | 0/1087 [00:00<?, ?it/s]

-0.029198646545410156


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x747b0c255210>> (for post_run_cell), with arguments args (<ExecutionResult object at 747b0c254430, execution_count=6 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 747b0c2545e0, raw_cell="# W&B 설정 및 학습 실행
wandb.init(project='CenterSpeed',.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/harry/ros2_ws/src/TinyCenterSpeed/src/train/train_CenterSpeed_wandb.ipynb#X15sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

In [ ]:
# 학습 곡선 시각화
plt.plot(losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training/Validation Loss Curve')
plt.legend()
plt.show()